# Titanic — Machine Learning from Disaster (v2)

**v1 result:** leaderboard accuracy 0.75837, OOF ensemble 0.8328.
**v2 changes vs v1:**
1. New features: FarePerPerson, TicketPrefix, AgeGroup, FamilySizeBin and
   a Sex x Pclass combination column.
2. Slightly stronger regularisation for the small data (891 rows):
   shallower trees and fewer estimators where it helps.
3. Blend choice is automatic: plain average vs OOF-accuracy-weighted
   average — whichever wins on out-of-fold accuracy is submitted.


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

DATA = '/kaggle/input/titanic'
if not os.path.exists(DATA):
    DATA = '/kaggle/input/competitions/titanic'
print('data dir:', DATA)
train = pd.read_csv(f'{DATA}/train.csv')
test = pd.read_csv(f'{DATA}/test.csv')
print('train:', train.shape, '| test:', test.shape)
print('survival rate: %.4f' % train['Survived'].mean())


In [ ]:
def prep(df, age_medians, fare_medians, embarked_mode):
    df = df.copy()
    df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')
    df['Title'] = df['Title'].replace({
        'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
        'Capt': 'Officer', 'Col': 'Officer', 'Major': 'Officer',
        'Dr': 'Officer', 'Rev': 'Officer', 'Jonkheer': 'Rare',
        'Don': 'Rare', 'Dona': 'Rare', 'Lady': 'Rare',
        'Countess': 'Rare', 'Sir': 'Rare'})
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['Deck'] = df['Cabin'].fillna('M').str[0]
    df['Age'] = df.apply(
        lambda r: age_medians.get((r['Sex'], r['Pclass']), np.nan)
        if pd.isna(r['Age']) else r['Age'], axis=1)
    df['Fare'] = df.apply(
        lambda r: fare_medians.get(r['Pclass'], np.nan)
        if pd.isna(r['Fare']) else r['Fare'], axis=1)
    df['Embarked'] = df['Embarked'].fillna(embarked_mode)
    df['FarePerPerson'] = df['Fare'] / df['FamilySize']
    df['TicketPrefix'] = df['Ticket'].str.extract(r'^([A-Za-z]+)')[0].fillna('NONE')
    df['AgeGroup'] = pd.cut(df['Age'], bins=[-1, 12, 18, 35, 60, 200],
                            labels=['Child', 'Teen', 'Adult', 'Mid', 'Senior'])
    df['FamilySizeBin'] = pd.cut(df['FamilySize'], bins=[0, 1, 4, 20],
                                 labels=['Alone', 'Small', 'Large'])
    df['Sex_Pclass'] = df['Sex'] + '_' + df['Pclass'].astype(str)
    return df

age_medians = train.groupby(['Sex', 'Pclass'])['Age'].median().to_dict()
fare_medians = train.groupby('Pclass')['Fare'].median().to_dict()
embarked_mode = train['Embarked'].mode()[0]
train = prep(train, age_medians, fare_medians, embarked_mode)
test = prep(test, age_medians, fare_medians, embarked_mode)

CATS = ['Sex', 'Embarked', 'Title', 'Deck', 'TicketPrefix', 'AgeGroup', 'FamilySizeBin', 'Sex_Pclass']
NUMS = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'FarePerPerson']
FEATURES = NUMS + CATS
X = pd.get_dummies(train[FEATURES], columns=CATS)
X_test = pd.get_dummies(test[FEATURES], columns=CATS).reindex(columns=X.columns, fill_value=0)
y = train['Survived']
print('n features:', X.shape[1])


In [ ]:
# ---- 5-fold stratified CV + blend picked by OOF accuracy ----
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = {k: np.zeros(len(X)) for k in ['lr', 'rf', 'lgb', 'xgb', 'cat']}
pred = {k: np.zeros(len(X_test)) for k in oof}

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    sc = StandardScaler().fit(X_tr)
    models = {
        'lr': LogisticRegression(max_iter=1000, C=1.0),
        'rf': RandomForestClassifier(n_estimators=400, max_depth=6, random_state=42, n_jobs=-1),
        'lgb': lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=15,
                                  min_child_samples=20, random_state=42, n_jobs=-1, verbose=-1),
        'xgb': xgb.XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=3,
                                 eval_metric='logloss', random_state=42, n_jobs=-1),
        'cat': CatBoostClassifier(iterations=400, learning_rate=0.05, depth=5,
                                  random_seed=42, verbose=0),
    }
    for k, mdl in models.items():
        if k == 'lr':
            mdl.fit(sc.transform(X_tr), y_tr)
            oof[k][va_idx] = mdl.predict_proba(sc.transform(X_va))[:, 1]
            pred[k] += mdl.predict_proba(sc.transform(X_test))[:, 1] / 5
        else:
            mdl.fit(X_tr, y_tr)
            oof[k][va_idx] = mdl.predict_proba(X_va)[:, 1]
            pred[k] += mdl.predict_proba(X_test)[:, 1] / 5

accs = {k: accuracy_score(y, (oof[k] > 0.5).astype(int)) for k in oof}
for k, v in accs.items():
    print(f'OOF acc {k}: {v:.4f}')
w = np.array([accs[k] for k in oof])
w = w / w.sum()
oof_plain = np.mean([oof[k] for k in oof], axis=0)
oof_weighted = np.sum([oof[k] * w[i] for i, k in enumerate(oof)], axis=0)
acc_plain = accuracy_score(y, (oof_plain > 0.5).astype(int))
acc_weighted = accuracy_score(y, (oof_weighted > 0.5).astype(int))
print(f'OOF acc PLAIN blend: {acc_plain:.4f} | WEIGHTED blend: {acc_weighted:.4f}')
USE_WEIGHTED = acc_weighted >= acc_plain
print('chosen blend:', 'WEIGHTED' if USE_WEIGHTED else 'PLAIN')


In [ ]:
# ---- Final submission (chosen blend) ----
if USE_WEIGHTED:
    test_pred = (np.sum([pred[k] * w[i] for i, k in enumerate(pred)], axis=0) > 0.5).astype(int)
else:
    test_pred = (np.mean([pred[k] for k in pred], axis=0) > 0.5).astype(int)
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': test_pred})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print('submission saved:', submission.shape)
print('predicted survivors:', int(submission['Survived'].sum()), 'of', len(submission))
